In [1]:
import pandas as pd
import datetime as dt
import numpy as np
from sqlalchemy import create_engine


In [2]:
USER = "root"
PASSWORD = "Abhi8383055393"
HOST = "localhost"
PORT = "3306"
DATABASE = "Sales_and_profitability_analysis"

conn_str = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"

mysql_conn = create_engine(conn_str)

In [ ]:
# orders detail table
orders = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/orders.csv")

orders['order_date'] = pd.to_datetime(orders['order_date']).dt.date

orders.info()

# order item table
order_items = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/order_items.csv")

order_items.info()

# Product Table
products = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/products.csv")

products.info()

products[products['category'].isna() == True].count()

products.loc[products['category'].isna(),['subcategory','category']]

products[products['subcategory'] == "Storage"].head(3)

category_map = {
"Printing": "Office Supplies",
"Accessories" : "Electronics",	
"Footwear" : "Clothing",	
"Audio" : "Electronics",	
"Cleaning" : "Home Appliances",	
"Paper" : "Office Supplies",	
"Desks" : "Furniture",	
"Mobiles" : "Electronics",	
"Stationery" : "Office Supplies",	
"Storage" : "Furniture"
}

products['category'] = products['category'].fillna(products['subcategory'].map(category_map))

products.info()

products['launch_date'] = pd.to_datetime(products['launch_date']).dt.date

products.info()

# region table 
regions = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/regions.csv")
regions

# sales channel table 
sales_channels = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/sales_channels.csv")
sales_channels

# returns 
returns = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/returns.csv")
returns['return_date'] = pd.to_datetime(returns['return_date']).dt.date
returns.info()


# target 
targets = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/targets.csv")
targets.info()


# export unclean data into mysql
order_items.to_sql("order_items",mysql_conn,if_exists='replace',index=False)
products.to_sql("products",mysql_conn,if_exists='replace',index=False)
returns.to_sql("returns",mysql_conn,if_exists='replace',index=False)
targets.to_sql("targets",mysql_conn,if_exists='replace',index=False)
regions.to_sql("regions",mysql_conn,if_exists='replace',index=False)
sales_channels.to_sql("sales_channels",mysql_conn,if_exists='replace',index=False)

order_items.columns

products.columns

# merging the tables 
full_order_details = pd.merge(order_items,products,how='right',on='product_id')


main_fact_table = pd.merge(
    full_order_details,
    orders,
    how='right',
    on='order_id'
)


main_fact_table.head()

main_fact_table.isna().value_counts()

main_fact_table['order_item_id'].isna().value_counts()

null_order_item_rows = main_fact_table[main_fact_table['order_item_id'].isna() == True]
null_order_item_rows

main_fact_table.isna().value_counts()

main_fact_table['product_id'].isna().value_counts()

main_fact_table[main_fact_table['product_id'].isna() == True]

main_fact_table = main_fact_table.dropna(subset=['order_item_id'])

main_fact_table.isna().value_counts()

order_items.shape



In [26]:
main_fact_table.columns


Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price',
       'discount_pct', 'unit_cost_x', 'product_name', 'category',
       'subcategory', 'brand', 'unit_cost_y', 'standard_price', 'supplier_id',
       'launch_date', 'order_date', 'customer_id', 'region_id', 'channel_id',
       'payment_method', 'order_status'],
      dtype='str')

In [58]:
main_fact_table['order_status'] = main_fact_table['order_status'].str.lower().str.replace(r"[^\w\s]", "", regex=True).str.replace(r"\s+", "_", regex=True)

In [27]:
main_fact_table.drop(columns=["unit_cost_x"], inplace=True)

In [34]:
main_fact_table.columns

Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price',
       'discount_pct', 'product_name', 'category', 'subcategory', 'brand',
       'unit_cost_y', 'standard_price', 'supplier_id', 'launch_date',
       'order_date', 'customer_id', 'region_id', 'channel_id',
       'payment_method', 'order_status', 'gross_price', 'discount_amt',
       'net_revenue'],
      dtype='str')

In [37]:
# creating important metrics 
main_fact_table['gross_price'] = main_fact_table['unit_price'] * main_fact_table['quantity']
main_fact_table['discount_amt'] = main_fact_table['gross_price'] * main_fact_table['discount_pct']
main_fact_table['net_revenue'] = main_fact_table['gross_price']  -  main_fact_table['discount_amt']
main_fact_table['cogs'] = main_fact_table['unit_cost_y'] * main_fact_table['quantity']
main_fact_table['gross_profit'] = main_fact_table['net_revenue'] -  main_fact_table['cogs']
main_fact_table['profit_margin'] = main_fact_table['gross_profit'] / main_fact_table['net_revenue']




In [ ]:
main_fact_table['order_status'].value_counts()

In [ ]:
main_fact_table[['discount_amt', 'net_revenue', 'gross_price']].head()

In [59]:
# "Give me the overall sales and profitability picture."
# KPI Baseline

# Total Revenue
total_revenue_in_each_status = main_fact_table.groupby('order_status')['net_revenue'].sum().sort_values(ascending=False)

# Total Gross Profit
gross_profit_by_order_status = main_fact_table.groupby('order_status')['gross_profit'].sum().sort_values(ascending=False)

# Profit Margin %
profit_margin_by_order_status = main_fact_table.groupby('order_status')['profit_margin'].sum().sort_values(ascending=False)

# Total Units Sold
unit_solds_by_category = main_fact_table.groupby('order_status')['quantity'].sum().sort_values(ascending=False)

# Total Orders
total_orders_by_category = main_fact_table.groupby('order_status')['order_id'].count().sort_values(ascending=False)

# Total Customers
total_customers_by_category = main_fact_table.groupby('order_status')['customer_id'].count().sort_values(ascending=False)

# Average Order Value
main_fact_table['avg_order_value'] = main_fact_table['net_revenue'] / main_fact_table['order_id'].count()
avg_order_value_by_order_status = main_fact_table.groupby('order_status')['avg_order_value'].sum().sort_values(ascending=False)

# Average Discount %
avg_discount_pct_by_order_status = main_fact_table.groupby('order_status')['discount_pct'].mean().sort_values(ascending=False)



In [57]:
unit_solds_by_category

order_status
completed             542450.0
partially returned     27979.0
returned               27908.0
cancelled              21922.0
pending                 9097.0
completed                 64.0
Name: quantity, dtype: float64